In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import h5py
import os
from pathlib import Path

In [3]:
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_1_2500.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_2501_5000.mat'
# mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_5001_7500.mat'
mat_file = r'E:\Thesis\thesis_code\data\oscilloscope\10Mbps\ethernet_packets_7501_10000.mat'


In [4]:
with h5py.File(mat_file, 'r') as f:
    reference_packet = np.array(f['packets'])  # shape: (recordLength,)
    # Access metadata
    
    metadata = f['metadata']
    sample_rate = metadata['sample_rate'][0][0]
    trigger_level = metadata['trigger_level'][0][0]
    record_length = metadata['record_length'][0][0]
    num_frames = metadata['num_frames'][0][0]

print(f"Sample Rate: {sample_rate} Hz")
print(f"Trigger Level: {trigger_level} V")
print(f"Record Length: {record_length} samples")
print(f"Number of Frames: {num_frames}")

Sample Rate: 1250000000.0 Hz
Trigger Level: -0.5 V
Record Length: 250000.0 samples
Number of Frames: 2500.0


In [5]:
len(reference_packet)

2500

### Extract Signal Region 
- Extract Signal Region and while keeping the packet lengths uniform

In [6]:
def extract_signal_region(signal, threshold=0.1):
    """Trims the quiet parts of the signal before and after the packet."""
    active = np.abs(signal) > threshold
    indices = np.where(active)[0]
    if indices.size == 0:
        return np.array([])
    return signal[indices[0] : indices[-1] + 1]

In [7]:
# time_axis = np.arange(record_length) / sample_rate
# for packet in reference_packet[:10]:
#     plt.figure(figsize=(12,6))
#     plt.plot(time_axis, packet, label='Packet')  # Plot each packet
#     # plt.title(f'Reference Ethernet Packet')
#     plt.xlabel('Time (s)')
#     plt.ylabel('Voltage (V)')
#     plt.grid(True)
#     plt.show()

In [8]:
samples_per_bit = int(sample_rate / 10e6)  # 10Mbps
samples_per_bit

125

In [9]:
ideal_packet_length = samples_per_bit*(8+148+4)*8
ideal_packet_length

160000

In [10]:
for packet in reference_packet[:10]:
    trimmed_packet = extract_signal_region(packet)
    print(f"Trimmed packet length: {len(trimmed_packet)} samples")

Trimmed packet length: 160140 samples
Trimmed packet length: 160202 samples
Trimmed packet length: 160203 samples
Trimmed packet length: 160203 samples
Trimmed packet length: 160142 samples
Trimmed packet length: 160140 samples
Trimmed packet length: 160204 samples
Trimmed packet length: 160203 samples
Trimmed packet length: 160141 samples
Trimmed packet length: 160203 samples


In [11]:
# count = 0
# max_packet_length = 0
# min_packet_length = float('inf')  # Start with max possible length
# for packet in reference_packet:
#     trimmed_packet = extract_signal_region(packet)
#     if len(trimmed_packet) < ideal_packet_length:
#         count += 1
#         continue
#     if len(trimmed_packet) > max_packet_length:
#         max_packet_length = len(trimmed_packet)
#     if len(trimmed_packet) < min_packet_length:
#         min_packet_length = len(trimmed_packet)
# count

In [12]:
# print(f"Max packet length: {max_packet_length} samples")
# print(f"Min packet length: {min_packet_length} samples")
# print(f"Number of packets shorter than ideal length: {count} out of {len(reference_packet)}")

In [13]:
def prepare_for_classification(extracted_signal, target_length=160250):
    """
    Pads the signal to target_length. 
    Raises ValueError if the signal is already longer than the target.
    """
    current_len = len(extracted_signal)

    if current_len > target_length:
        raise ValueError(
            f"Packet length ({current_len}) exceeds target_length ({target_length}). "
            "Increase target_length or check for extraction errors/noise."
        )

    if current_len < target_length:
        # Pad with zeros at the end (Tail Padding)
        pad_size = target_length - current_len
        return np.pad(extracted_signal, (0, pad_size), mode='constant')
    
    return extracted_signal

In [14]:
# for packet in reference_packet:
#     trimmed_packet = extract_signal_region(packet)
#     if len(trimmed_packet) < ideal_packet_length:
#         continue  # Skip packets that are too short
#     try:
#         prepared_packet = prepare_for_classification(trimmed_packet, target_length=160250)
#         print(f"Prepared packet length: {len(prepared_packet)} samples")
#     except ValueError as e:
#         print(e)
    

In [ ]:
# def replace_data_with_h5(mat_file_path, count = None):
#     path = Path(mat_file_path)
#     parts = list(path.parts)

#     # Replace 'data' with 'h5_data'
#     parts = ["h5_data" if p == "data" else p for p in parts[:-1]]
#     if count is None:
#         return str(Path(*parts, path.stem + ".h5"))
#     return str(Path(*parts, path.stem + f"_{count}.h5"))

In [ ]:
output_tmp = replace_data_with_h5(mat_file)  # Start with count=0 for temporary file
# Extract directory
dir_path = os.path.dirname(output_tmp)

# Create directory if it doesn't exist
os.makedirs(dir_path, exist_ok=True)

# Abort if file already exists
if os.path.exists(output_tmp):
    raise FileExistsError(f"File already exists: {output_tmp}")
target_length = 160250

count = 0

with h5py.File(output_tmp, "w") as f:
    dset = f.create_dataset(
        "signals",
        shape=(0, target_length),
        maxshape=(None, target_length),
        dtype=np.float32,
        chunks=True,
        compression="gzip",
        compression_opts=9
    )

    for packet in reference_packet:
        trimmed_packet = extract_signal_region(packet)

        if len(trimmed_packet) < ideal_packet_length:
            continue

        try:
            prepared_packet = prepare_for_classification(
                trimmed_packet,
                target_length=target_length
            )

            prepared_packet = np.asarray(prepared_packet, dtype=np.float32)

            dset.resize(count + 1, axis=0)
            dset[count] = prepared_packet

            count += 1
            print(f"Stored packet {count}")

        except ValueError as e:
            print(e)

    # store metadata inside file (VERY important)
    f.attrs["num_signals"] = count
    f.attrs["target_length"] = target_length



# rename AFTER writing
final_name = replace_data_with_h5(mat_file, count)
os.rename(output_tmp, final_name)

print("Saved as:", final_name)

Stored packet 1
Stored packet 2
Stored packet 3
Stored packet 4
Stored packet 5
Stored packet 6
Stored packet 7
Stored packet 8
Stored packet 9
Stored packet 10
Stored packet 11
Stored packet 12
Stored packet 13
Stored packet 14
Stored packet 15
Stored packet 16
Stored packet 17
Stored packet 18
Stored packet 19
Stored packet 20
Stored packet 21
Stored packet 22
Stored packet 23
Stored packet 24
Stored packet 25
Stored packet 26
Stored packet 27
Stored packet 28
Stored packet 29
Stored packet 30
Stored packet 31
Stored packet 32
Stored packet 33
Stored packet 34
Stored packet 35
Stored packet 36
Stored packet 37
Stored packet 38
Stored packet 39
Stored packet 40
Stored packet 41
Stored packet 42
Stored packet 43
Stored packet 44
Stored packet 45
Stored packet 46
Stored packet 47
Packet length (163889) exceeds target_length (160250). Increase target_length or check for extraction errors/noise.
Stored packet 48
Stored packet 49
Stored packet 50
Stored packet 51
Stored packet 52
Stored pa

Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x000001F111D5D300>
Traceback (most recent call last):
  File "C:\Users\NITRO 5\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):

KeyboardInterrupt: 


Stored packet 189
Stored packet 190
Stored packet 191
Stored packet 192
Stored packet 193
Stored packet 194
Stored packet 195
Stored packet 196
Stored packet 197
Stored packet 198
Stored packet 199
Stored packet 200
Stored packet 201
Stored packet 202
Stored packet 203
Stored packet 204
Stored packet 205
Stored packet 206
Stored packet 207
Stored packet 208
Stored packet 209
Stored packet 210
Stored packet 211
Stored packet 212
Stored packet 213
Stored packet 214
Stored packet 215
Stored packet 216
Stored packet 217
Stored packet 218
Stored packet 219
Stored packet 220
Stored packet 221
Stored packet 222


In [ ]:
def extract_time_domain_features(signal):
    """Extract time-domain statistical features"""
    features = {}
    
    # Basic statistical moments
    features['std'] = np.std(signal)
    features['skewness'] = skew(signal)
    features['rms'] = np.sqrt(np.mean(signal**2))
    features['mean'] = np.mean(signal)
    features['kurtosis'] = kurtosis(signal)
    features['peak'] = np.max(np.abs(signal))
    
    # Shape factors
    features['shape_factor'] = features['rms'] / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) != 0 else 0
    features['impulse_factor'] = features['peak'] / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) != 0 else 0
    features['crest_factor'] = features['peak'] / features['rms'] if features['rms'] != 0 else 0
    features['clearance_factor'] = features['peak'] / (np.mean(np.sqrt(np.abs(signal)))**2) if np.mean(np.sqrt(np.abs(signal))) != 0 else 0
    
    return features

In [ ]:
def extract_frequency_domain_features(signal, fs):
    """Extract frequency-domain features"""
    # Compute FFT
    fft_signal = fft(signal)
    freqs = fftfreq(len(signal), 1/fs)
    
    # Get positive frequencies only
    positive_freq_idx = freqs > 0
    fft_positive = fft_signal[positive_freq_idx]
    freqs_positive = freqs[positive_freq_idx]
    
    # Find top 10 harmonic components
    magnitudes = np.abs(fft_positive)
    top_indices = np.argsort(magnitudes)[-10:]
    
    features = {}
    for i, idx in enumerate(top_indices):
        features[f'freq_mag_{i}'] = magnitudes[idx]
        features[f'freq_val_{i}'] = freqs_positive[idx]
    
    return features